In [ ]:
# Import relevant packages.

import matplotlib.pyplot as plt
import numpy as np
import math
import scipy.integrate as integrate
import scipy.special as special
from scipy.interpolate import interp1d
import matplotlib.ticker
from scipy.integrate import quad
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
from matplotlib.ticker import FormatStrFormatter
import pandas as pd
from array import *
matplotlib.rcParams.update(matplotlib.rcParamsDefault)
matplotlib.rcParams['mathtext.fontset'] = 'stix'
#matplotlib.rcParams['font.family'] = 'STIXGeneral'
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['image.origin'] = 'lower'
import matplotlib.pyplot as plt


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize


#define parameters/ constants in cgs units
c=3.0*1e10 #speed of light
H_0=7.2*1e6 #hubble's constant
h=6.63*1e-27 #Planck's constant
k_b=1.38*1e-16 #Boltzmann's constant
Omega_M=0.27 #matter density fraction
Omega_Alpha=0.73 #Dark energy density fraction
D_H=c/H_0*3.086*1e24 #hubble distance


# Functions used to redshift SEDs

# convert individual string in the format of 'x.xxxD+xx' to decimal number
def strtonum(strD):
    dot=1  #position of dot
    digit=99999 #position of D
    Num=0.0 #converted number
    sign=0 #position of the sign
    for i in range(len(strD)):
        if strD[i]=='+':
            sign=1;
        if strD[i]=='-':
            sign=-1
        if i==0:
            Num+=int(float(strD[i]))
        if strD[i]=='D' or strD[i]=='E':
            digit=i
        if sign==0:
            if i>dot and i<digit:
                Num+=int(strD[i])*10**(-(i-1))
        elif i>digit+1 and sign==1:
                Num*=10**(int(strD[i])*10**(len(strD)-i-1))
        elif i>digit+1 and sign==-1:
                Num*=10**(-int(strD[i])*10**(len(strD)-i-1))
    return Num

# convert to an array
def convertdata(data):
    array=[]
    for i in range(len(data)):
        array.insert(i,strtonum(data[i]))
    return array

#convert freq at emission to observed lambda(in microns) since we want to plot flux vs.wavelength
def freqtolam(data,z):
    freq=convertdata(data[0:,0])
    Lambda_obs=[]
    for i in range(len(freq)):
        if c/freq[i]*(1+z)*1e4<=20:
            Lambda_obs.insert(i,c/freq[i]*(1+z)*1e4)
    return Lambda_obs

# Redshift the SED, flux is in units of nJy
def shftspecTLU(data,z,R):
    freq=convertdata(data[0:,0]) #import the converted data
    flux=convertdata(data[0:,1])
    #eddfactor=data[0:,2]
    if z!=0:
        I = integrate.quad(lambda x: D_H*1/math.sqrt(Omega_M*(1+x)**3+Omega_Alpha), 0, z)
        DC_z=I[0]-I[1]
        DL_z=(1+z)*DC_z
        fluxobs=[]
        for i in range(len(freqtolam(data,z))):
            fluxobs.insert(i,4*np.pi*flux[i]*(1+z)*R**2/DL_z**2*1e32)
            #follow eq8 of: https://academic.oup.com/mnras/article/422/3/2164/1043351, if source z>0
        #flux=list(map(lambda x: ,Lambda))
    else:
        fluxobs=[]
        for i in range(len(freqtolam(data,z))):
            fluxobs.insert(i,4*np.pi*flux[i]*1e32)
    return fluxobs



In [ ]:
#log_10_R_AU = 0.1979*log10(M/M_0)+ 0.6
#log_10_T_eff =  0.14262 * log_10_(M/M_o) + 3.4297
import numpy as np
def T_eff(solar_mass):
  T_eff = 10**(0.1326255 * np.log10(solar_mass) + 3.483)
  return T_eff

def log_g(solar_mass):
  R_AU = 10**(0.194265* np.log10(solar_mass) + 0.6197384)
  R_cm = R_AU*1.496*10**13
  M = solar_mass *2*10**33
  G = 6.678 *10**-8
  g = (G*M)/R_cm**2
  log_g = np.log10(g)
  return log_g
def R_cm(solar_mass):
  R_AU = 10**(0.194265* np.log10(solar_mass) + 0.6197384)
  R_cm = R_AU*1.496*10**13

  return R_cm
'''
with open('surf_grav_Temp_AC.csv', 'w') as f:
    #np.savetxt(f, [wavelength_band], delimiter=',')
    for i in np.arange(5,7,0.05):
      array = [T_eff(10**i), log_g(10**i), R_cm(10**i), i, 10**i]
      np.savetxt(f, [array], delimiter=',')
'''

print(T_eff(6*10**5))
print(log_g(6*10**5))
print(R_cm(7*10**5))

In [ ]:
def nanojansky_to_ergs_per_cm2_per_A(njy, wavelength_micron):
    # Constants
    speed_of_light_cm_per_s = 29979245800  # Speed of light in cm/s

    # Convert nanoJansky to Jansky
    jy = njy * 1e-9  # 1 nanoJansky = 10^-9 Jansky

    # Convert Jansky to erg/s/cm²/Hz
    flux_density_hz = jy * 1e-23  # 1 Jansky = 10^-23 erg/s/cm²/Hz

    # Convert wavelength from Angstroms to centimeters
    wavelength_cm = wavelength_micron * 1e-4  # 1 Angstrom = 10^-8 cm

    # Convert erg/s/cm²/Hz to erg/s/cm²/Angstrom
    flux_density_angstrom = flux_density_hz * (speed_of_light_cm_per_s / (wavelength_cm**2))

    return flux_density_angstrom*10**-8*(10**21)

nanojansky_to_ergs_per_cm2_per_A(14, 5.99)

In [ ]:
obs_spec = pd.read_csv('JZ11_spec.csv')
lower_bound = 1
upper_bound = 5.29

# Remove entries where the column's values are between the specified range (including NaN values)
obs_spec = obs_spec[obs_spec['Wavelength']>1]
obs_spec = obs_spec[obs_spec['Wavelength']<5.29]

obs_lambda = obs_spec['Wavelength']
print(obs_spec.columns)
obs_flux = obs_spec['Flux']
#obs_unc = obs_spec['Flux Error']*1e21
obs_unc = np.abs(0.2*obs_flux)


upper = obs_flux + obs_unc
lower = obs_flux - obs_unc
print(obs_unc)
'''
obs_spec2 = pd.read_csv('jades_z11_1dspectrum.csv')

lower_bound2 = 1
upper_bound2 = 5.29

# Remove entries where the column's values are between the specified range (including NaN values)
obs_spec2 = obs_spec2[obs_spec2['Wavelength']>1]
obs_spec2 = obs_spec2[obs_spec2['Wavelength']<5.29]

obs_lambda2 = obs_spec2['Wavelength']
print(obs_spec2.columns)
obs_flux2 = obs_spec2['Flux']*1e21
obs_unc2 = obs_spec2['Flux Error']*1e21
#obs_unc = np.abs(0.2*obs_flux)


upper2 = obs_flux2 + obs_unc2
lower2 = obs_flux2 - obs_unc2
print(obs_unc2)
'''




# Plot the data
plt.figure(figsize=(10, 6))
plt.plot(obs_lambda, obs_flux, label='Observed Spectrum')
#plt.plot(obs_lambda2, obs_flux2, label='Observed Spectrum')
#plt.fill_between(obs_lambda, lower, upper, color='gray', alpha=0.5)
#plt.fill_between(obs_lambda2, lower2, upper2, color='gray', alpha=0.5)
plt.xlim(1,5)
#plt.ylim(-1.5, 2)
plt.axvline(0.1640*12.37)
plt.show()
obs_lambda = np.array(obs_lambda)


def interpolate_template_spec(data, mu, z, radius, min_wavelegnth, max_wavelength, obs_lambda):
    Spectrum_T3M6_31e5 = pd.read_csv(data
                                        ,header=None,sep='\s+')
    data_T3M6_31e5=Spectrum_T3M6_31e5.to_numpy()
    flux_8 = freqtolam(data_T3M6_31e5,z)
    spec_8 = shftspecTLU(data_T3M6_31e5, z, radius)
    flux_8 = np.array(flux_8)
    spec_8 = mu*np.array(spec_8)
    # Constants
    #print(flux_8)
    #print(spec_8)
    spec_now = np.zeros(len(flux_8))
    for i in range(len(flux_8)):
      spec_now[i] = nanojansky_to_ergs_per_cm2_per_A(spec_8[i], flux_8[i])
    indices = np.where((flux_8 >= min_wavelegnth) & (flux_8 <= max_wavelength))[0]

    # Step 2: Subset array1 using these indices
    subset_lambda = flux_8[indices]

    # Step 3: Use the same indices to subset array2
    subset_spec = spec_now[indices]
    lya_shift = 0.1216*(1+z)
    lam_cut = []
    flx_cut = []

    index = 0
    for i in subset_lambda:
        if i >lya_shift:
            lam_cut.append(i)
            flx_cut.append(1*subset_spec[index])
            index = index + 1
        else:
            lam_cut.append(i)
            flx_cut.append(0)

            index = index + 1
    lam_cut = np.array(lam_cut)
    flx_cut = np.array(flx_cut)

    interpolation_function = interp1d(lam_cut, flx_cut, kind='linear', bounds_error=False, fill_value='extrapolate')

    new_spec = interpolation_function(obs_lambda)

    #main_array = np.array([subset_lambda, subset_spec, new_spec])

    return new_spec
'''
spec_array = interpolate_template_spec('T3M6_0e5.13', 1, 11.3, 8.21e+14, 0.009, 5.26, obs_lambda)


#plt.plot(spec_array[0], spec_array[1])
#plt.plot(spec_array[0], spec_array[2])
plt.plot(obs_lambda, spec_array)
plt.plot(obs_lambda, obs_flux)
print(np.mean((spec_array - obs_flux)**2))
plt.show()
'''

In [ ]:
Mass_10_5 = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30, 40, 50, 60, 70, 80, 90])
#Radius = np.array([5.83, 6.67, 7.22, 7.64, 7.98, 8.26, 8.51, 8.74, 8.94, 9.12])
data = np.array(['T3M1_0e5.13', 'T3M2_0e5.13', 'T3M3_0e5.13', 'T3M4_0e5.13', 'T3M5_0e5.13', 'T3M6_0e5.13', 'T3M7_0e5.13', 'T3M8_0e5.13', 'T3M9_0e5.13', 'T3M1_0e6.13', 'ACM2_00e+06.13', 'ACM3_00e+06.13', 'ACM4_00e+06.13', 'ACM5_00e+06.13', 'ACM6_00e+06.13', 'ACM7_00e+06.13', 'ACM8_00e+06.13', 'ACM9_00e+06.13'])
print(data[13])

In [ ]:
#Radius-Mass relationship for AC

Masses = np.log10(np.array([10**5, 10**6, 10**7]))#in terms of 10^5
Radius = np.log10(np.array([5.83e+14, 9.12e+14, 1.45e+15]))
log_g = np.log10(10**np.array([1.591, 2.202, 2.8]))
temp = np.log10(np.array([14000, 19000, 27000]))
interpolation_mass_radii = interp1d(Masses, Radius, kind='linear', fill_value="extrapolate")
interpolation_mass_g = interp1d(Masses, log_g, kind='linear', fill_value="extrapolate")
interpolation_mass_temp = interp1d(Masses, temp, kind='linear', fill_value="extrapolate")

#new_mass = np.linspace(10**6, 10**7, 10)
new_mass = 16.7*10**5
new_radii = 10**(interpolation_mass_radii(np.log10(new_mass)))
new_log_g = interpolation_mass_g(np.log10(new_mass))
new_temp = 10**interpolation_mass_temp(np.log10(new_mass))


print(new_temp)
print(new_log_g)
print(new_radii)
temp_base = np.array([14000, 19000, 27000])
mass_base = np.array([10**5, 10**6, 10**7])

plt.scatter(new_mass, new_temp)
plt.scatter(mass_base, temp_base)
plt.show()

# Create a DataFrame from the arrays
data1 = {
    'mass': new_mass,
    'radius': new_radii,
    'log_g': new_log_g,
    'temp': new_temp
}
df = pd.DataFrame(data1)

# Convert the DataFrame to a CSV file
csv_filename = 'AC_10_6_7.csv'
df.to_csv(csv_filename, index=False)

print(f"Data successfully saved to {csv_filename}")


'''
new_masses = np.linspace(10**6,10**7, 10)
new_radii = interpolation_mass_radii(new_masses)
new_radii = interpolation_mass_radii(new_masses)
plt.scatter(Masses, Radius)
plt.plot(new_masses, new_radii)
plt.show()
print(new_radii)
'''

In [ ]:
from sklearn.metrics import mean_squared_error
obs_spec = pd.read_csv('JZ11_spec.csv')
obs_spec = obs_spec[obs_spec['Wavelength']>1]
obs_spec = obs_spec[obs_spec['Wavelength']<5.29]
obs_lambda = obs_spec['Wavelength'].values
obs_flux = (obs_spec[' Flux'].values)
#obs_error = (obs_spec['Flux Error'].values)*1e21
obs_error = np.abs(0.2*(obs_spec[' Flux'].values))
print(np.var(obs_flux))


import seaborn as sns
import matplotlib.pyplot as plt
def sampler_obs(obs_data, unc):
  n_sim = 100
  main_array = []
  for i in range(n_sim):
    sub_array = []
    for j in range(len(obs_data)):
      upper = obs_data[j] + unc[j]
      lower = obs_data[j] - unc[j]
      mean = obs_data[j]
      sigma = (upper - lower)/3.92
      sample = np.random.normal(mean, sigma)
      sub_array.append(sample)
    main_array.append(sub_array)

  return main_array

main_array = np.array(sampler_obs(obs_flux, obs_error))
print(main_array)
'''
plt.hist(main_array[:,0])
plt.show()
'''




mass_array = []
lensing_factor = []
mean_squared = []
redshift = []
for i in range(len(main_array[:,0])):
  array_obs = main_array[i,:]
  def obj_func(x):
    z = x[1]
    mass = x[0]
    mu =x[2]
    min_wavelength = 1
    max_wavelength = 5.28
    #R_cm = 10**(0.194265* np.log10(mass*10**5) + 0.6197384)*1.496*10**13
    index_of_nearest = np.argmin(np.abs(Mass_10_5 - mass))
    #print(index_of_nearest)
    main_data = data[index_of_nearest]
    mass_new = Mass_10_5[index_of_nearest]
    #print(main_data)
    R_cm = 10**interpolation_mass_radii(np.log10(mass_new * 10**5))
    #print(R_cm)
    template_array = np.array(interpolate_template_spec(main_data, mu, z, R_cm, min_wavelength, max_wavelength, obs_lambda))
    mse = (mean_squared_error(array_obs, template_array))
    return mse
  result = scipy.optimize.minimize(obj_func, [6, 11, 1], method='Nelder-Mead')
  minimized_val = result.x
  mse_value = result.fun
  print(i)
  print(minimized_val[0], minimized_val[1],  minimized_val[2], mse_value)

  mass_array.append(minimized_val[0])
  #lensing_factor.append(minimized_val[0])
  redshift.append(minimized_val[1])
  lensing_factor.append(minimized_val[2])
  mean_squared.append(mse_value)


